In [1]:
import json
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
BASE_DIR     = Path(".")
TEXTS_DIR    = BASE_DIR / "data_texts"
FR_PATH      = TEXTS_DIR / "20000lieues_fr.txt"
JSON_PATH    = BASE_DIR / "Points-escales-chapitres.json"
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

TEST_ESCALES = ["Île de Crespo", "Archipel des Pomotou", "Méditerranée"]

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
fr_text_full = fr_text_full.replace("\r\n", "\n").replace("\r", "\n")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ {len(fr_text_full):,} caractères chargés")
print(f"✅ {len(data)} escales dans le JSON")

# =========================================================
# EXTRACTION DU CHAPITRE
# =========================================================
ROMAINS_FR = {
    1:"PREMIER",2:"II",3:"III",4:"IV",5:"V",6:"VI",7:"VII",
    8:"VIII",9:"IX",10:"X",11:"XI",12:"XII",13:"XIII",14:"XIV",
    15:"XV",16:"XVI",17:"XVII",18:"XVIII",19:"XIX",20:"XX",
    21:"XXI",22:"XXII",23:"XXIII",24:"XXIV"
}

def extraire_chapitre(texte, partie, numero):
    romain = ROMAINS_FR.get(numero, str(numero))
    marqueur = f"CHAPITRE {romain}"
    split = texte.split("FIN DE LA PREMIÈRE PARTIE")
    zone = split[0] if partie == 1 else (split[1] if len(split) > 1 else texte)
    idx = zone.find(marqueur)
    if idx == -1:
        return None
    idx_fin = zone.find("CHAPITRE", idx + len(marqueur))
    return zone[idx:idx_fin].strip() if idx_fin != -1 else zone[idx:].strip()

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 1500}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8")).get("response", "").strip()
    except Exception as e:
        return f"Erreur : {e}"

def prompt_especes(nom_escale, texte):
    return f"""Lis ce texte extrait de "Vingt mille lieues sous les mers" (escale : {nom_escale}).

TEXTE :
{texte[:10000]}

Donne uniquement la liste des espèces maritimes mentionnées dans ce texte.
Une espèce par ligne, précédée d'un tiret.
Pas d'explication, pas de commentaire, pas de latin, pas de catégorie.
Si aucune espèce n'est mentionnée, écris : "Aucune espèce mentionnée."
"""



# =========================================================
# TEST SUR 3 ESCALES
# =========================================================
for escale in data:
    nom_fr = escale.get("Escales du Nautilus", {}).get("fr", "")
    if nom_fr not in TEST_ESCALES:
        continue
    ch_info = escale.get("chapitre")
    if not ch_info:
        continue

    partie = ch_info["partie"]
    numero = ch_info["chapitre"]
    ch_fr = extraire_chapitre(fr_text_full, partie, numero)

    print(f"\n🧭 {nom_fr} — Partie {partie}, Ch.{numero} ({len(ch_fr) if ch_fr else 0} car.)")
    print("🤖 Envoi à Ollama...")

    reponse = appeler_ollama(prompt_especes(nom_fr, ch_fr)) if ch_fr else "Chapitre introuvable"

    display(Markdown(f"### 🧭 {nom_fr}\n{reponse}\n---"))
    time.sleep(2)

print("\n✅ Test terminé !")


✅ 908,396 caractères chargés
✅ 32 escales dans le JSON

🧭 Île de Crespo — Partie 1, Ch.15 (16671 car.)
🤖 Envoi à Ollama...


### 🧭 Île de Crespo
- Cladostèphes verticillées
- Padines-paon
- Caulerpes à feuilles de vigne
- Callithamnes granifères
- Délicates céramies à teintes écarlates
- Agares disposées en éventails
- Acétabules
- Varechs
---


🧭 Archipel des Pomotou — Partie 1, Ch.19 (21056 car.)
🤖 Envoi à Ollama...


### 🧭 Archipel des Pomotou
- munérophis
- ostreidae (huîtres)
- ostrea lamellosa
- aucune espèce mentionnée
---


🧭 Méditerranée — Partie 2, Ch.7 (22108 car.)
🤖 Envoi à Ollama...


### 🧭 Méditerranée
- Lamproie
- Oxyrhynque
- Raie (non spécifiée)
- Squal milandre
- Renard marin
- Dorade (genre spare)
- Esturgeon magnifique
- Scombre-thon
- Gymnote fierasfer blanc
- Murene congres
- Gade merlu
- Cœpole ténia
- Trygle
- Trygle hirondelle
- Holocentre méron
- Alose
- Turbot
- Mullus rouget
- Limande
- Flé
- Plie
- Sole
- Carrelet
- Aucune espèce mentionnée (entre les blennies, surmulets, labres, éperlan, exocet, anchois, pagel, bogue et orphe)
- Phoque (moine)
- Tortue large de six pieds
- Cacouanne à carapace allongée
- Galéolaire orangée
---


✅ Test terminé !
